# 03 · Broadcast events with a fanout exchange

**Level:** beginner · **Time:** 25–35 minutes · **Prerequisites:** notebooks 01–02.

Build an order-event broadcaster with separate analytics and email queues. See that each queue receives
a copy, then demonstrate that an unbound subscriber does not receive old events retroactively.

```text
                         ┌→ analytics queue → analytics consumer
producer → fanout exchange
                         └→ email queue → email consumer
```

A fanout exchange routes a message to its bound queues. Routing keys do not select subscribers for
this exchange type. Two consumers on one queue compete; two different bound queues receive copies.


In [ ]:
from pathlib import Path
import sys
import json
import time
import pika

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course" / "rabbit.py").is_file()), None)
assert ROOT is not None, "Open Jupyter from the RabbitMQ course folder."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message


## 1. Declare the exchange and two subscriber queues

These are named, non-exclusive lab queues so they remain available between short connection scopes.
An exclusive temporary subscriber queue would disappear when its owning connection closes.


In [ ]:
exchange = unique_name("events.fanout")
analytics_queue = unique_name("analytics")
email_queue = unique_name("email")
late_queue = unique_name("late.subscriber")
with connect() as connection:
    channel = connection.channel()
    channel.exchange_declare(exchange=exchange, exchange_type="fanout", durable=False)
    for queue in [analytics_queue, email_queue]:
        channel.queue_declare(queue=queue, durable=False)
        channel.queue_bind(exchange=exchange, queue=queue, routing_key="")
print("Exchange:", exchange)
print("Subscriber queues:", analytics_queue, email_queue, sep="\n")


## 2. Publish one event and inspect both copies

Predict the total number of deliveries: one published event becomes one delivery in each bound queue.
The two acknowledgement tags belong to their deliveries; neither subscriber acknowledges for the other.


In [ ]:
event = {"event_id": "event-001", "type": "order.created", "order_id": 42}
with connect() as connection:
    channel = connection.channel()
    channel.basic_publish(exchange=exchange, routing_key="ignored.by.fanout",
                          body=json.dumps(event).encode(),
                          properties=pika.BasicProperties(content_type="application/json"))
    deliveries = {}
    for subscriber, queue in [("analytics", analytics_queue), ("email", email_queue)]:
        method, _, body = wait_for_message(channel, queue)
        deliveries[subscriber] = json.loads(body)
        channel.basic_ack(method.delivery_tag)
    assert deliveries == {"analytics": event, "email": event}
    print(json.dumps(deliveries, indent=2))


## 3. Add a late subscriber

A queue starts receiving routed events when its binding exists. An exchange is not an event-history
database. Predict whether the new queue contains `event-001` before we publish another event.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    channel.queue_declare(queue=late_queue, durable=False)
    channel.queue_bind(exchange=exchange, queue=late_queue, routing_key="")
    assert channel.basic_get(queue=late_queue, auto_ack=False)[0] is None
    second_event = {"event_id": "event-002", "type": "order.created", "order_id": 43}
    channel.basic_publish(exchange=exchange, routing_key="",
                          body=json.dumps(second_event).encode())
    for queue in [analytics_queue, email_queue, late_queue]:
        method, _, body = wait_for_message(channel, queue)
        assert json.loads(body)["event_id"] == "event-002"
        channel.basic_ack(method.delivery_tag)
    print("The late subscriber saw only the second event.")


## 4. Exercise: unsubscribe email without deleting its queue

Remove the email binding using `queue_unbind`, publish one more event, and prove analytics and the
late subscriber receive it while email stays empty. Removing a binding prevents future routing through
that binding; it does not erase messages already in the queue.


In [ ]:
# Suggested solution. The previous cell emptied all three queues.
with connect() as connection:
    channel = connection.channel()
    channel.queue_unbind(exchange=exchange, queue=email_queue, routing_key="")
    channel.basic_publish(exchange=exchange, routing_key="", body=b"event-003")
    for queue in [analytics_queue, late_queue]:
        method, _, body = wait_for_message(channel, queue)
        assert body == b"event-003"
        channel.basic_ack(method.delivery_tag)
    assert channel.basic_get(queue=email_queue, auto_ack=False)[0] is None
    print("Email unsubscribed; both other subscriptions received the event.")


## 5. Clean up and discuss

Delete each queue and then this lesson's exchange. Design question: to let three email workers share
the email workload, would you create three email queues or three consumers on the existing email queue?
**Answer:** three consumers on the same queue share deliveries. Three bound queues would create three copies.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    for queue in [analytics_queue, email_queue, late_queue]:
        channel.queue_delete(queue=queue)
    channel.exchange_delete(exchange=exchange)
print("Fanout resources removed.")


**Troubleshooting:** no delivery usually means the queue was not bound before publishing. A fanout
exchange with no matching bound queues can discard a publication; publisher confirms plus mandatory
routing detection are covered later. A consumer being offline differs from its queue not existing:
an existing non-exclusive queue can accumulate events while its consumer is offline.

Official reference: [RabbitMQ publish/subscribe tutorial](https://www.rabbitmq.com/tutorials/tutorial-three-python).
